# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guided workflow for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. It demonstrates best practices for referencing data elements by their `@id` and performing exploratory analysis.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
* [`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print metadata overview
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset @id: {metadata.id}")
print(f"Version: {getattr(metadata, 'version', 'n/a')}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

Croissant datasets define record sets via the `recordSet` property.

**Tip:** All entities are referenced by their `@id`.


In [ ]:
# List all record sets by @id
record_sets = []
if hasattr(metadata, 'recordSet'):
    # recordSet can be a list or empty
    if isinstance(metadata.recordSet, list):
        for rs in metadata.recordSet:
            if hasattr(rs, 'id'):
                record_sets.append(rs.id)
            elif isinstance(rs, dict) and '@id' in rs:
                record_sets.append(rs['@id'])
    elif isinstance(metadata.recordSet, dict):
        record_sets.append(metadata.recordSet['@id'])

if not record_sets:
    # Fallback: mlcroissant auto-discovers record sets from the schema
    record_sets = [rs.id for rs in dataset.record_sets]

print("Available record sets (@id):")
for rs_id in record_sets:
    print(f"- {rs_id}")

# For each record set, list fields
for rs in dataset.record_sets:
    print(f"\nRecord set @id: {rs.id}")
    print("Fields:")
    for field in rs.fields:
        print(f"  - {field.id}: {field.name} (type: {getattr(field, 'dataType', 'n/a')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we extract all record sets dynamically, load them into pandas DataFrames (referenced by their `@id`), and preview their columns.

In [ ]:
# Extract data from each record set
dataframes = {}
for rs in dataset.record_sets:
    rs_id = rs.id
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nRecord set @id: {rs_id}")
    print("DataFrame columns:", df.columns.tolist())

# If there is at least one record set, preview the first
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"\nPreview records from record set @id: {main_rs_id}")
    display(dataframes[main_rs_id].head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, grouping. Operations reference columns by their `@id`.

Let's choose the first numeric field from the main record set for demonstration.

In [ ]:
# Automatically discover a numeric field (type Integer or Float) from the record set
main_rs = dataset.record_sets[0] if dataset.record_sets else None
numeric_field_id = None
numeric_types = ['Integer', 'Float', 'Number']

if main_rs:
    for field in main_rs.fields:
        dtype = getattr(field, 'dataType', '').split(':')[-1]  # Handles schema:Integer
        if dtype in numeric_types:
            numeric_field_id = field.id
            break

main_rs_id = main_rs.id if main_rs else None
df = dataframes.get(main_rs_id, pd.DataFrame())

if numeric_field_id and not df.empty:
    # Filter: greater than a threshold (use median as threshold if available)
    if numeric_field_id in df.columns:
        threshold = df[numeric_field_id].median() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping by a categorical field
        group_field_id = None
        for field in main_rs.fields:
            dtype = getattr(field, 'dataType', '').split(':')[-1]
            if dtype == 'Text' and field.id != numeric_field_id:
                group_field_id = field.id
                break
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
            print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions and relationships. Plots reference fields by their `@id`. Here we plot the numeric field distribution and (if available) grouped means.

In [ ]:
# Plot numeric field distribution
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=20, edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping was performed
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(8, 4))
        plt.bar(grouped_df[group_field_id], grouped_df[numeric_field_id])
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
The FAIR^2 dataset for adoption predictors in rangeland management was successfully loaded, explored, and processed using `mlcroissant`. All entity references utilized their `@id` for clarity and schema consistency.

**Key insights:**
- Metadata clearly describes socio-demographics and knowledge interventions.
- We reviewed record sets and fields by `@id`, dynamically loaded all tables.
- EDA highlighted numeric field distributions and possible groupings.

For further analysis, refer to the Croissant schema and documentation for available record sets and their fields.